## 1 · Imports

Lädt alle externen Bibliotheken, die im Notebook verwendet werden.

| Gruppe | Bibliothek | Zweck |
|---|---|---|
| Standardbibliothek | `pathlib.Path` | Plattformunabhängige Dateipfade |
| Standardbibliothek | `time`, `logging` | Wartezeiten zwischen API-Aufrufen, strukturierte Logs |
| Datenverarbeitung | `pandas` | Tabellen (DataFrames), Parquet lesen/schreiben |
| Datenverarbeitung | `pyarrow` | Parquet-Dateiformat (Backend für pandas) |
| HTTP & API | `requests` | HTTP-Anfragen an die Binance REST API |
| Visualisierung | `matplotlib` | Diagramme (für spätere Analyse) |

In [23]:
# ── Standardbibliotheken ──────────────────────────────────────
from pathlib import Path
import time
import gc
import random
import logging

# ── Datenverarbeitung ─────────────────────────────────────────
import numpy as np
import pandas as pd
import pyarrow

# ── Visualisierung ────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# ── HTTP & API ────────────────────────────────────────────────
import requests
import ccxt

# ──  Konfiguration ──────────────────────────────
import hmac
import hashlib
import configparser
from dataclasses import dataclass


import shutil
import json

## 2 · Logging-Konfiguration

Richtet das Python-Logging ein, das alle Statusmeldungen während der Datenbeschaffung ausgibt.

Das Format `line: %(lineno)-4s %(levelname)-8s %(message)s` zeigt bei jeder Meldung:
- die **Zeilennummer** im Code (hilfreich zur Fehlersuche)
- das **Level** (`INFO` = normale Information, `WARNING` = unvollständige Daten oder übersprungene Symbole)
- die eigentliche **Nachricht**

`force=True` stellt sicher, dass die Konfiguration auch bei erneutem Ausführen der Zelle überschrieben wird. Die Variable `log` wird im gesamten Notebook als einheitliche Schnittstelle für alle Ausgaben verwendet.

In [24]:
logging.basicConfig(
    level=logging.INFO,
    format="line: %(lineno)-4s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)


## 3 · Pfade & Konfiguration

Definiert alle Verzeichnisse und Parameter, die das Notebook benötigt. Alle Pfade werden relativ zum Projektstamm (`PROJECT_ROOT`) aufgebaut, damit das Notebook auf jedem Rechner ohne Anpassung funktioniert.

**Wichtige Pfade:**
- `ASSETS_PATH`  Liste aller Coins (aus CoinGecko, 45 Assets)
- `ASSET_DAILY_PATH`  Tägliche Marktdaten (Market Cap, Volumen) pro Asset
- `ASSET_OHLCV_5M_DIR`  Zielordner für die OHLCV-Parquet-Dateien

**API-Parameter:**
- `INTERVAL = "5m"`  Jede Kerze deckt 5 Minuten ab → 288 Kerzen pro Tag
- `BINANCE_KLINE_LIMIT = 1000`  Maximale Kerzen pro API-Anfrage (Binance-Limit)
- `SLEEP_BETWEEN_REQUESTS = 0.05`  50 ms Pause zwischen Anfragen, um Rate-Limits zu vermeiden

`controlpath()` prüft beim Start, ob alle erwarteten Dateien und Ordner vorhanden sind, und gibt deren Inhalt aus.

In [ ]:
# Shared paths — from config.py
import sys
sys.path.insert(0, str(Path("..").resolve()))
from config import (
    PROJECT_ROOT, DATA_DIR, RAW_DIR, FETCHED_DIR,
    PROCESSED_DIR, FEATURES_DIR, SRC_PATH, SECRET_PATH,
    ASSETS_PATH,
    ASSET_DAILY_DIR as ASSET_DAILY_PATH,   # ohlcv reads entire directory via .glob()
    OHLCV_DIR as ASSET_OHLCV_5M_DIR,
)

TARGET_VOLUME_EXCHANGE = "binance"

# API parameters — stay here because they are fetch-specific
INTERVAL            = "5m"
BINANCE_KLINE_LIMIT = 1000

def controlpath():
    log.info(f"Projektverzeichnis:            {PROJECT_ROOT}")
    log.info(f"Datenverzeichnis:              {DATA_DIR}")
    log.info(f"Rohdatenverzeichnis:           {RAW_DIR}")
    log.info(f"Gefetchte Daten:               {FETCHED_DIR}")
    log.info(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    log.info(f"Source-Code:                   {SRC_PATH}")
    log.info(f"Secret:                        {SECRET_PATH}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-Daily-Verzeichnis":   ASSET_DAILY_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        log.info(f"  {symbol}  {name}: {path}")

        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    log.info(f"       📄 {f.name}")
            else:
                log.info(f"       ⚠️  Ordner ist leer — keine .parquet Dateien!")


OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

controlpath()

## 4 · OHLCV-Datenbeschaffung

Dieses Block enthält die gesamte Logik zum Herunterladen der Kerzendaten von Binance. Er besteht aus vier Funktionen, die nacheinander aufgerufen werden:

### `load_valid_binance_symbols()`
Fragt einmalig die Binance-API nach allen aktiven Handelssymbolen (`/api/v3/exchangeInfo`). Symbole die auf CoinGecko gelistet sind, aber auf Binance nicht existieren (z. B. `USDTUSDT`, `HYPEUSDT`, `PYUSDUSDT`), werden damit automatisch herausgefiltert.

### `load_ticker_list()`
Liest die lokalen Parquet-Dateien (`assets.parquet`, `asset_daily.parquet`) und baut ein Dictionary auf:
```
tickersperday = { datum → DataFrame mit Ticker-Liste für diesen Tag }
```
Pro Tag gibt es einen Eintrag mit den ~45 Coins die an diesem Tag in der Top-Liste waren.

### `json_to_parquet()`
Speichert die heruntergeladenen Rohdaten als Parquet-Datei. Dabei werden die Daten nach Jahr und Monat aufgeteilt (`ohlcv_2026-01.parquet`, `ohlcv_2026-02.parquet`, …). Bereits vorhandene Einträge werden mit `drop_duplicates` nicht doppelt gespeichert.

### `fetch_ohlcv_for_month()`  Hauptfunktion
Steuert den gesamten Ablauf:
1. Lädt die Ticker-Liste und gültige Binance-Symbole
2. Iteriert pro **Monat** (nicht pro Tag) über die Daten
3. Prüft für jeden Monat welche Tage bereits vollständig gespeichert sind
    Ein Tag gilt als vollständig wenn der **Median** aller Ticker ≥ 270 Kerzen hat
4. Fetcht nur die fehlenden Tage/Ticker von Binance
5. Speichert die Ergebnisse via `json_to_parquet()`

**Steuerungsvariable:**

| `FORCE_REFETCH` | Verhalten |
|---|---|
| `False` | Normalbetrieb  vollständige Monate werden sofort übersprungen |
| `True` | Reparatur-Modus  jeden Ticker pro Tag einzeln prüfen und nur fehlende nachfetchen |


![UML](../../ohlcv_UML.png)

In [34]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
SLEEP_BETWEEN_REQUESTS = 0.05

# False  →  nur fehlende/unvollständige Tage fetchen (Normalbetrieb)
# True   →  jeden Ticker pro Tag prüfen und fehlende nachfetchen (Reparatur-Modus)
FORCE_REFETCH = True

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def load_valid_binance_symbols() -> set:
    """Einmalig alle gültigen Spot-Symbole von Binance laden."""
    info = SESSION.get(f"{BASE_URL}/api/v3/exchangeInfo", timeout=20).json()
    return {s["symbol"] for s in info["symbols"] if s["status"] == "TRADING"}

VALID_BINANCE_SYMBOLS: set = set()   # wird in fetch_ohlcv_for_month befüllt

def public_get(path: str, params: dict | None = None):
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            df_all = (
                pd.concat([pd.read_parquet(out), month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Aktualisiert: {out.name}  →  {len(df_all):,} Zeilen")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Erstellt: {out.name}  →  {len(df_all):,} Zeilen")

        df_all.to_parquet(out, index=False)


# =========================
# Ticker-Liste laden
# =========================
def load_ticker_list() -> dict:
    log.info("─" * 60)
    log.info(" SCHRITT 1/3 — Assets laden")
    log.info("─" * 60)
    df_assets = pd.read_parquet(ASSETS_PATH)
    log.info(f"  → {len(df_assets)} Assets geladen")

    log.info("\n" + "─" * 60)
    log.info(" SCHRITT 2/3 — Daily-Snapshots laden")
    log.info("─" * 60)
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    log.info(f"  → {[f.name for f in parquet_files]}")
    df_daily = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    df_daily["date"] = pd.to_datetime(df_daily["date"]).dt.date
    log.info(f"  → Shape: {df_daily.shape}")

    log.info("\n" + "─" * 60)
    log.info(" SCHRITT 3/3 — Pro Tag: Ticker-Liste aufbauen")
    log.info("─" * 60)
    tickersperday = {}
    for date in sorted(df_daily["date"].unique()):
        df_date = df_daily[df_daily["date"] == date].copy()
        df = df_date.merge(df_assets[["asset_id", "symbol", "name"]], on="asset_id", how="left")
        df["market_symbol"] = df["symbol"].str.upper() + "USDT"
        tickersperday[date] = df[[
            "asset_id", "symbol", "name", "market_symbol",
            "market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"
        ]].reset_index(drop=True)

    log.info(f"  → {len(tickersperday)} Tage aufgebaut")
    return tickersperday


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_month() -> None:
    mode = "REPARATUR" if FORCE_REFETCH else "NORMAL"
    log.info("=" * 60)
    log.info(f"  OHLCV-DATENBESCHAFFUNG GESTARTET  [{mode}]")
    log.info("=" * 60)

    tickersperday = load_ticker_list()

    global VALID_BINANCE_SYMBOLS
    VALID_BINANCE_SYMBOLS = load_valid_binance_symbols()
    log.info(f"  → {len(VALID_BINANCE_SYMBOLS)} gültige Binance-Symbole geladen")

    seen_months = set()
    for snapshot_date, df_tickers in sorted(tickersperday.items()):
        year  = snapshot_date.year
        month = snapshot_date.month
        if (year, month) in seen_months:
            continue
        seen_months.add((year, month))

        all_symbols = df_tickers["market_symbol"].tolist()
        invalid     = [s for s in all_symbols if s not in VALID_BINANCE_SYMBOLS]
        symbols     = [s for s in all_symbols if s in VALID_BINANCE_SYMBOLS]
        if invalid:
            log.warning(f"  ⚠️  {len(invalid)} Symbole nicht auf Binance — werden übersprungen: {invalid}")
        month_start   = pd.Timestamp(year=year, month=month, day=1)
        days_in_month = month_start.days_in_month
        all_dates     = pd.date_range(month_start, periods=days_in_month, freq="D").date

        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        # Vorhandene Daten einlesen: Kerzen pro (Tag, Ticker)
        existing_dates  = set()
        ticker_candles  = {}   # (date, market_symbol) → Kerzenanzahl
        if out.exists():
            df_ex         = pd.read_parquet(out, columns=["timestamp", "market_symbol"])
            df_ex["date"] = pd.to_datetime(df_ex["timestamp"]).dt.date
            per_ticker    = df_ex.groupby(["date", "market_symbol"]).size()
            ticker_candles = per_ticker.to_dict()
            day_median    = per_ticker.groupby("date").median()
            existing_dates = set(day_median[day_median >= 270].index)
            incomplete     = set(day_median[day_median < 270].index)
            if incomplete:
                log.warning(f"  ⚠️  Unvollständige Tage: {sorted(incomplete)}")

        missing_dates = [d for d in all_dates if d not in existing_dates]

        log.info(f"\n{'=' * 60}")
        log.info(f"  Monat {year}-{month:02d}:  {days_in_month} Tage  |  {len(symbols)} Ticker")
        log.info(f"  Vorhanden ({len(existing_dates)}) / Fehlend ({len(missing_dates)}): {missing_dates}")
        log.info(f"{'=' * 60}")

        if not FORCE_REFETCH:
            # Normalbetrieb: nur komplett fehlende Tage fetchen
            if not missing_dates:
                log.info(f"  ✅ Alle Tage vollständig — Monat wird übersprungen")
                continue
            dates_to_process = missing_dates
        else:
            # Reparatur-Modus: alle Tage durchgehen, pro Ticker prüfen
            log.info(f"  ⚠️  REPARATUR-MODUS: alle {days_in_month} Tage werden geprüft")
            dates_to_process = list(all_dates)

        fetch_errors = []

        for current_date in dates_to_process:
            # Welche Ticker brauchen noch Daten für diesen Tag?
            if FORCE_REFETCH:
                symbols_to_fetch = [
                    s for s in symbols
                    if ticker_candles.get((current_date, s), 0) < 270
                ]
                if not symbols_to_fetch:
                    log.info(f"  [✔] {current_date} — alle Ticker vollständig")
                    continue
                log.info(f"\n  ── {current_date}: {len(symbols_to_fetch)}/{len(symbols)} Ticker fehlen ──")
            else:
                symbols_to_fetch = symbols
                log.info(f"\n  ── Fetche {current_date} ({len(symbols)} Ticker) ──")

            all_rows = []
            day_errors = []

            for market_symbol in symbols_to_fetch:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000) - 1,
                        "limit":     BINANCE_KLINE_LIMIT,
                    })
                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })
                    time.sleep(SLEEP_BETWEEN_REQUESTS)
                except Exception as e:
                    msg = f"{current_date} | {market_symbol}: {e}"
                    day_errors.append(msg)
                    fetch_errors.append(msg)

            log.info(f"  → {len(all_rows):,} Kerzen  |  {len(day_errors)} Fehler")
            for err in day_errors:
                log.warning(f"    [FEHLER] {err}")

            if all_rows:
                json_to_parquet(all_rows)
            else:
                log.info(f"  [SKIP] Keine Daten für {current_date}")

        if FORCE_REFETCH and fetch_errors:
            log.warning(f"\n  ❌ Reparatur {year}-{month:02d}: {len(fetch_errors)} Fehler gesamt:")
            for err in fetch_errors:
                log.warning(f"    {err}")

    log.info("\n" + "=" * 60)
    log.info("  OHLCV-DATENBESCHAFFUNG ABGESCHLOSSEN")
    log.info("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_month()


line: 105   INFO      ============================================================
line: 106   INFO        OHLCV-DATENBESCHAFFUNG GESTARTET  [REPARATUR]
line: 107   INFO      ============================================================
line: 68    INFO      ────────────────────────────────────────────────────────────
line: 69    INFO       SCHRITT 1/3 — Assets laden
line: 70    INFO      ────────────────────────────────────────────────────────────
line: 72    INFO        → 45 Assets geladen
line: 74    INFO      
────────────────────────────────────────────────────────────
line: 75    INFO       SCHRITT 2/3 — Daily-Snapshots laden
line: 76    INFO      ────────────────────────────────────────────────────────────
line: 78    INFO        → ['asset_daily.parquet']
line: 81    INFO        → Shape: (4013, 6)
line: 83    INFO      
────────────────────────────────────────────────────────────
line: 84    INFO       SCHRITT 3/3 — Pro Tag: Ticker-Liste aufbauen
line: 85    INFO      ───────────

## 5 · Qualitätsprüfung  `ohlcvCheck()`

Liest alle gespeicherten OHLCV-Parquet-Dateien und gibt eine Übersichtstabelle aus.

**Spalten der Tabelle:**
- `Ticker`  Anzahl unterschiedlicher Symbole an diesem Tag
- `Kerzen`  Gesamtanzahl Kerzen (Ticker × Kerzen/Ticker)
- `Kerzen/Ticker`  Durchschnittliche Kerzen pro Symbol

**Bewertung pro Tag:**
- ✅ ≥ 274 Kerzen/Ticker  vollständig (288 = 24 h × 12 Intervalle à 5 min)
- ⚠️ 200–273 Kerzen/Ticker  teilweise vorhanden
- ❌ < 200 Kerzen/Ticker  unvollständig, sollte neu gefetcht werden

Diese Zelle hat **keinen Einfluss** auf die gespeicherten Daten  sie dient nur zur Kontrolle.

In [31]:
# =========================
# OHLCV Schnellübersicht
# =========================
def ohlcvCheck():
    log.info("=" * 70)
    log.info("OHLCV DATEI-ÜBERSICHT")
    log.info("=" * 70)

    for file in sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet")):
        df = pd.read_parquet(file)
        df["date"] = pd.to_datetime(df["timestamp"]).dt.date

        day_ticker  = df.groupby("date")["market_symbol"].nunique()
        day_candles = df.groupby("date").size()

        log.info(f"\n📄 {file.name}")
        log.info(f"   Tage gesamt : {df['date'].nunique()}")
        log.info(f"   Ticker Ø    : {day_ticker.mean():.0f}  (min {day_ticker.min()}, max {day_ticker.max()})")
        print()
        log.info(f"   {'Datum':<14} {'Ticker':>8} {'Kerzen':>10} {'Kerzen/Ticker':>15}")
        log.info(f"   {'-'*14} {'-'*8} {'-'*10} {'-'*15}")

        for date in sorted(day_ticker.index):
            ticker  = day_ticker[date]
            candles = day_candles[date]
            ratio   = candles / ticker if ticker > 0 else 0
            status  = "✅" if ratio >= 274 else "⚠️ " if ratio >= 200 else "❌"
            log.info(f"   {str(date):<14} {ticker:>8} {candles:>10,} {ratio:>14.0f}  {status}")

    log.info("\n" + "=" * 70)
ohlcvCheck()

line: 5     INFO      ======================================================================
line: 6     INFO      OHLCV DATEI-ÜBERSICHT
line: 7     INFO      ======================================================================
line: 16    INFO      
📄 ohlcv_2026-01.parquet
line: 17    INFO         Tage gesamt : 31
line: 18    INFO         Ticker Ø    : 40  (min 40, max 40)
line: 20    INFO         Datum            Ticker     Kerzen   Kerzen/Ticker
line: 21    INFO         -------------- -------- ---------- ---------------
line: 28    INFO         2026-01-01           40     11,520            288  ✅
line: 28    INFO         2026-01-02           40     11,520            288  ✅
line: 28    INFO         2026-01-03           40     11,520            288  ✅
line: 28    INFO         2026-01-04           40     11,520            288  ✅
line: 28    INFO         2026-01-05           40     11,520            288  ✅
line: 28    INFO         2026-01-06           40     11,520            288  ✅
l

line: 16    INFO      
📄 ohlcv_2026-02.parquet
line: 17    INFO         Tage gesamt : 28
line: 18    INFO         Ticker Ø    : 40  (min 40, max 40)
line: 20    INFO         Datum            Ticker     Kerzen   Kerzen/Ticker
line: 21    INFO         -------------- -------- ---------- ---------------
line: 28    INFO         2026-02-01           40     11,520            288  ✅
line: 28    INFO         2026-02-02           40     11,520            288  ✅
line: 28    INFO         2026-02-03           40     11,520            288  ✅
line: 28    INFO         2026-02-04           40     11,520            288  ✅
line: 28    INFO         2026-02-05           40     11,520            288  ✅
line: 28    INFO         2026-02-06           40     11,520            288  ✅
line: 28    INFO         2026-02-07           40     11,520            288  ✅
line: 28    INFO         2026-02-08           40     11,520            288  ✅
line: 28    INFO         2026-02-09           40     11,520            288 

line: 16    INFO      
📄 ohlcv_2026-03.parquet
line: 17    INFO         Tage gesamt : 31
line: 18    INFO         Ticker Ø    : 40  (min 40, max 41)
line: 20    INFO         Datum            Ticker     Kerzen   Kerzen/Ticker
line: 21    INFO         -------------- -------- ---------- ---------------
line: 28    INFO         2026-03-01           40     11,520            288  ✅
line: 28    INFO         2026-03-02           40     11,520            288  ✅
line: 28    INFO         2026-03-03           40     11,520            288  ✅
line: 28    INFO         2026-03-04           40     11,520            288  ✅
line: 28    INFO         2026-03-05           40     11,520            288  ✅
line: 28    INFO         2026-03-06           40     11,520            288  ✅
line: 28    INFO         2026-03-07           40     11,520            288  ✅
line: 28    INFO         2026-03-08           40     11,520            288  ✅
line: 28    INFO         2026-03-09           40     11,520            288 

## 6 · Zwischenergebnis-Inspektion

Zeigt alle Zwischenstufen der Datenpipeline im Detail an  nur zur Analyse und zum Verständnis des Datenflusses. Diese Zelle verändert keine Daten und muss nicht bei jedem Lauf ausgeführt werden.

**Dargestellte Stufen:**

| Stufe | Inhalt |
|---|---|
| `df_assets` | 45 Coins aus CoinGecko mit Metadaten |
| `parquet_files` | Liste der Daily-Snapshot-Dateien |
| `tickersperday` | Dictionary: Datum → Ticker-DataFrame |
| `existing_dates` | Set der bereits vollständig gespeicherten Tage |
| `raw` | Rohe Binance API-Antwort (Liste von 12-elementigen Arrays) |
| `all_rows` | Umgewandelte Dicts mit sprechenden Spaltennamen |
| `df` | Pandas DataFrame bereit zum Speichern |
| Parquet-Dateien | Finales Ergebnis mit Shape, Ticker- und Tagesanzahl |

In [35]:
# =============================================================
# ZWISCHENERGEBNIS-INSPEKTION — alle Stufen aus dem Diagramm
# =============================================================

SEP = "=" * 65

# ── 1. df_assets ─────────────────────────────────────────────
print(f"\n{SEP}")
print("1) df_assets   (aus assets.parquet)")
print(SEP)
df_assets = pd.read_parquet(ASSETS_PATH)
print(f"  Typ    : {type(df_assets)}")
print(f"  Shape  : {df_assets.shape}  →  {df_assets.shape[0]} Assets, {df_assets.shape[1]} Spalten")
print(f"  Spalten: {df_assets.columns.tolist()}")
print(f"  Dtypes :\n{df_assets.dtypes.to_string()}")
print("\n  Erste 3 Zeilen:")
display(df_assets.head(3))

# ── 2. parquet_files ─────────────────────────────────────────
print(f"\n{SEP}")
print("2) parquet_files   (Daily-Snapshot Dateien)")
print(SEP)
parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
print(f"  Typ    : {type(parquet_files)}  →  Liste von Path-Objekten")
print(f"  Anzahl : {len(parquet_files)}")
for f in parquet_files:
    print(f"    📄 {f.name}")

# ── 3. tickersperday ─────────────────────────────────────────
print(f"\n{SEP}")
print("3) tickersperday   (Dict: date → DataFrame)")
print(SEP)
_tpd = {}
_df_daily = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
_df_daily["date"] = pd.to_datetime(_df_daily["date"]).dt.date
for _date in sorted(_df_daily["date"].unique()):
    _df_date = _df_daily[_df_daily["date"] == _date].copy()
    _df = _df_date.merge(df_assets[["asset_id", "symbol", "name"]], on="asset_id", how="left")
    _df["market_symbol"] = _df["symbol"].str.upper() + "USDT"
    _tpd[_date] = _df[["asset_id", "symbol", "name", "market_symbol",
                         "market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"]].reset_index(drop=True)

_example_date = sorted(_tpd)[0]
print(f"  Typ      : {type(_tpd)}")
print(f"  Einträge : {len(_tpd)}  (ein Eintrag pro Snapshot-Tag)")
print(f"  Keys     : {sorted(_tpd)[:3]} … {sorted(_tpd)[-1]}")
print(f"\n  Beispiel-Eintrag für {_example_date}:")
print(f"    Typ    : {type(_tpd[_example_date])}")
print(f"    Shape  : {_tpd[_example_date].shape}")
print(f"    Spalten: {_tpd[_example_date].columns.tolist()}")
display(_tpd[_example_date].head(5))

# ── 4. existing_dates ────────────────────────────────────────
print(f"\n{SEP}")
print("4) existing_dates   (Set: bereits gespeicherte Tage)")
print(SEP)
_out_jan = ASSET_OHLCV_5M_DIR / "ohlcv_2026-01.parquet"
if _out_jan.exists():
    _df_ex = pd.read_parquet(_out_jan, columns=["timestamp"])
    _df_ex["date"] = pd.to_datetime(_df_ex["timestamp"]).dt.date
    _existing = set(_df_ex["date"].unique())
else:
    _existing = set()
print(f"  Typ    : {type(_existing)}")
print(f"  Anzahl : {len(_existing)} Tage bereits vorhanden (Beispiel: Jan 2026)")
print(f"  Inhalt : {sorted(_existing)[:5]} …")

# ── 5. raw   (ein echter Binance-API Aufruf) ─────────────────
print(f"\n{SEP}")
print("5) raw   (Binance API Antwort — JSON Liste)")
print(SEP)
_raw = public_get("/api/v3/klines", params={
    "symbol":    "BTCUSDT",
    "interval":  "5m",
    "startTime": int(pd.Timestamp("2026-01-01", tz="UTC").timestamp() * 1000),
    "endTime":   int(pd.Timestamp("2026-01-01 00:30", tz="UTC").timestamp() * 1000),
    "limit":     6,
})
print(f"  Typ             : {type(_raw)}")
print(f"  Anzahl Kerzen   : {len(_raw)}")
print(f"  Elemente pro Kerze: {len(_raw[0])}")
print(f"\n  Erste Kerze (rohe Werte):")
_labels = ["open_time","open","high","low","close","volume","close_time",
           "quote_volume","trades","taker_buy_base","taker_buy_quote","ignore"]
for label, val in zip(_labels, _raw[0]):
    print(f"    [{_labels.index(label)}] {label:<20} = {val}")

# ── 6. all_rows   (nach Umwandlung in Dicts) ─────────────────
print(f"\n{SEP}")
print("6) all_rows   (Liste von Dictionaries)")
print(SEP)
_all_rows = []
for _row in _raw:
    _all_rows.append({
        "timestamp":     pd.to_datetime(int(_row[0]), unit="ms", utc=True),
        "market_symbol": "BTCUSDT",
        "open":          float(_row[1]),
        "high":          float(_row[2]),
        "low":           float(_row[3]),
        "close":         float(_row[4]),
        "volume_base":   float(_row[5]),
        "volume_quote":  float(_row[7]),
        "trade_count":   int(_row[8]),
    })
print(f"  Typ             : {type(_all_rows)}")
print(f"  Anzahl Einträge : {len(_all_rows)}")
print(f"  Keys eines Eintrags: {list(_all_rows[0].keys())}")
print(f"\n  Erster Eintrag:")
for k, v in _all_rows[0].items():
    print(f"    {k:<20} = {v}  ({type(v).__name__})")

# ── 7. df = pd.DataFrame(all_rows) ──────────────────────────
print(f"\n{SEP}")
print("7) df = pd.DataFrame(all_rows)   (Tages-DataFrame)")
print(SEP)
_df_day = pd.DataFrame(_all_rows)
print(f"  Typ    : {type(_df_day)}")
print(f"  Shape  : {_df_day.shape}  →  {_df_day.shape[0]} Kerzen, {_df_day.shape[1]} Spalten")
print(f"  Dtypes :\n{_df_day.dtypes.to_string()}")
print("\n  Alle Zeilen:")
display(_df_day)

# ── 8. Finale Parquet-Dateien ────────────────────────────────
print(f"\n{SEP}")
print("8) parquet files pro Monat   (Endresultat)")
print(SEP)
_ohlcv_files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
print(f"  Anzahl Dateien: {len(_ohlcv_files)}")
for _f in _ohlcv_files:
    _df_f = pd.read_parquet(_f)
    print(f"\n  📄 {_f.name}")
    print(f"     Shape  : {_df_f.shape}  →  {_df_f.shape[0]:,} Kerzen, {_df_f.shape[1]} Spalten")
    print(f"     Ticker : {_df_f['market_symbol'].nunique()}")
    print(f"     Tage   : {pd.to_datetime(_df_f['timestamp']).dt.date.nunique()}")
    print(f"     Von    : {_df_f['timestamp'].min()}")
    print(f"     Bis    : {_df_f['timestamp'].max()}")

print(f"\n{SEP}")
print("INSPEKTION ABGESCHLOSSEN")
print(SEP)



1) df_assets   (aus assets.parquet)
  Typ    : <class 'pandas.core.frame.DataFrame'>
  Shape  : (45, 8)  →  45 Assets, 8 Spalten
  Spalten: ['asset_id', 'coingecko_id', 'symbol', 'name', 'circulating_supply', 'total_supply', 'max_supply', 'top_exchange_count']
  Dtypes :
asset_id                int64
coingecko_id           object
symbol                 object
name                   object
circulating_supply    float64
total_supply          float64
max_supply            float64
top_exchange_count    float64

  Erste 3 Zeilen:


,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply,top_exchange_count
0,1,bitcoin,BTC,Bitcoin,2.001524e+07,2.001524e+07,21000000.0,19.0
1,2,ethereum,ETH,Ethereum,1.206910e+08,1.206910e+08,NaN,19.0
2,3,tether,USDT,Tether,1.844363e+11,1.899033e+11,NaN,19.0



2) parquet_files   (Daily-Snapshot Dateien)
  Typ    : <class 'list'>  →  Liste von Path-Objekten
  Anzahl : 1
    📄 asset_daily.parquet

3) tickersperday   (Dict: date → DataFrame)
  Typ      : <class 'dict'>
  Einträge : 90  (ein Eintrag pro Snapshot-Tag)
  Keys     : [datetime.date(2026, 1, 1), datetime.date(2026, 1, 2), datetime.date(2026, 1, 3)] … 2026-03-31

  Beispiel-Eintrag für 2026-01-01:
    Typ    : <class 'pandas.core.frame.DataFrame'>
    Shape  : (45, 7)
    Spalten: ['asset_id', 'symbol', 'name', 'market_symbol', 'market_cap_usd', 'global_volume_24h_usd', 'binance_volume_24h_usd']


,asset_id,symbol,name,market_symbol,market_cap_usd,global_volume_24h_usd,binance_volume_24h_usd
0,1,BTC,Bitcoin,BTCUSDT,1420084757787,2.863769e+10,None
1,2,ETH,Ethereum,ETHUSDT,263776706806,1.287374e+10,None
2,3,USDT,Tether,USDTUSDT,184451817798,4.660400e+10,None
3,4,BNB,BNB,BNBUSDT,81546829453,8.422890e+08,None
4,5,XRP,XRP,XRPUSDT,81445184554,1.776721e+09,None



4) existing_dates   (Set: bereits gespeicherte Tage)
  Typ    : <class 'set'>
  Anzahl : 31 Tage bereits vorhanden (Beispiel: Jan 2026)
  Inhalt : [datetime.date(2026, 1, 1), datetime.date(2026, 1, 2), datetime.date(2026, 1, 3), datetime.date(2026, 1, 4), datetime.date(2026, 1, 5)] …

5) raw   (Binance API Antwort — JSON Liste)
  Typ             : <class 'list'>
  Anzahl Kerzen   : 6
  Elemente pro Kerze: 12

  Erste Kerze (rohe Werte):
    [0] open_time            = 1767225600000
    [1] open                 = 87648.21000000
    [2] high                 = 87701.91000000
    [3] low                  = 87632.74000000
    [4] close                = 87701.91000000
    [5] volume               = 18.44460000
    [6] close_time           = 1767225899999
    [7] quote_volume         = 1616915.59764780
    [8] trades               = 4776
    [9] taker_buy_base       = 11.75992000
    [10] taker_buy_quote      = 1030903.69066620
    [11] ignore               = 0

6) all_rows   (Liste von Dicti

,timestamp,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count
0,2026-01-01 00:00:00+00:00,BTCUSDT,87648.21,87701.91,87632.74,87701.91,18.44460,1.616916e+06,4776
1,2026-01-01 00:05:00+00:00,BTCUSDT,87701.91,87816.00,87701.90,87764.06,57.78717,5.071411e+06,10412
2,2026-01-01 00:10:00+00:00,BTCUSDT,87764.06,87764.07,87742.61,87742.61,9.86578,8.657513e+05,2065
3,2026-01-01 00:15:00+00:00,BTCUSDT,87742.62,87760.19,87742.61,87760.19,9.47001,8.310241e+05,2382
4,2026-01-01 00:20:00+00:00,BTCUSDT,87760.19,87774.00,87760.18,87773.97,4.62578,4.060062e+05,1941
5,2026-01-01 00:25:00+00:00,BTCUSDT,87773.98,87776.19,87768.50,87776.18,10.01792,8.792995e+05,1745



8) parquet files pro Monat   (Endresultat)
  Anzahl Dateien: 3

  📄 ohlcv_2026-01.parquet
     Shape  : (357120, 9)  →  357,120 Kerzen, 9 Spalten
     Ticker : 40
     Tage   : 31
     Von    : 2026-01-01 00:00:00+00:00
     Bis    : 2026-01-31 23:55:00+00:00

  📄 ohlcv_2026-02.parquet
     Shape  : (322560, 9)  →  322,560 Kerzen, 9 Spalten
     Ticker : 40
     Tage   : 28
     Von    : 2026-02-01 00:00:00+00:00
     Bis    : 2026-02-28 23:55:00+00:00

  📄 ohlcv_2026-03.parquet
     Shape  : (358680, 9)  →  358,680 Kerzen, 9 Spalten
     Ticker : 41
     Tage   : 31
     Von    : 2026-03-01 00:00:00+00:00
     Bis    : 2026-03-31 23:55:00+00:00

INSPEKTION ABGESCHLOSSEN
